### mac 연산 구현

In [142]:
import json
import time
Filt_A = [[0,1,0],
          [1,1,1],
          [0,1,0]]
Filt_B = [[0,1,0],
          [1,1,1],
          [0,1,0]]
Filt_C = [[1,0,0],
          [1,0,1],
          [0,1,0]]
EPSILON = 1e-9

In [143]:
def mac(pattern, filt):
    add = 0
    for i in range(len(pattern)):
        for j in range(len(pattern[0])):
            mul = pattern[i][j]*filt[i][j]
            add += mul
    return add

def judge(score_a, score_b):
    if abs(score_a-score_b) < EPSILON:
        return "UNDECIDED"
    elif score_a > score_b:
        return "A"
    else: 
        return "B"
    
def measure_mac(pattern, filt, repeat=10):
    result = mac(pattern, filt)
    whole_time = 0
    for _ in range(repeat):
        start = time.perf_counter()
        mac(pattern, filt)
        end = time.perf_counter()
        elapsed_time = end - start
        whole_time += elapsed_time
    return result, ((whole_time)/repeat*1000) # ms 단위로 변환

In [8]:
def load_json(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

In [ ]:
def split(line, delim=" "): # delimiter를 기준으로 문자열을 나누어 리스트로 반환
    result = []
    src = ""
    for i in line:
        if i == delim:
            if src != "":
                result.append(src)
                src = ""
        else:
            src += i
    if src != "":
        result.append(src)
    return result

In [58]:
LABEL_MAP = {
    "+": "Cross",
    "cross": "Cross",
    "x": "X",
}

def normalize_label(label):
    label = label.replace(" ", "").lower()
    if label in LABEL_MAP:
        return LABEL_MAP[label]
    return None

def get_size(key):
    try:
        split_key = split(key, "_")
        return int(split_key[1])
    except (ValueError, IndexError):
        return None

In [115]:
data = load_json("data.json")

In [144]:
def normalize_filters(data):
    new_filters = {}
    for size_key, filter_group in data["filters"].items():
        new_size = get_size(size_key)
        if new_size is not None:
            new_group = {}
            for label, matrix in filter_group.items():
                new_label = normalize_label(label)
                if new_label is not None:
                    new_group[new_label] = matrix
            new_filters[new_size] = new_group
            labels_text = ", ".join(new_group.keys())
            
            if "Cross" in new_group and "X" in new_group:
                print(f"✓ {size_key} 필터 로드 완료 ({labels_text})")
            else:
                print(f"⚠ {size_key} 필터 일부 누락 ({labels_text})")
    return new_filters
            

In [145]:
filters = normalize_filters(data)


✓ size_5 필터 로드 완료 (Cross, X)
✓ size_13 필터 로드 완료 (Cross, X)
✓ size_25 필터 로드 완료 (Cross, X)


In [ ]:
key = "size_5_1"
pattern = data["patterns"][key]

# ① 키에서 크기
size = get_size(key)

# ② 그 크기 필터 꺼내기
cross_filt = filters[size]["Cross"]
x_filt = filters[size]["X"]

# ③ 크기 검증 (일단 생략, 다음 단계에서)

# ④ MAC 두 번
cross_score = mac(pattern["input"], cross_filt)
x_score = mac(pattern["input"], x_filt)

# ⑤ 판정
result = judge(cross_score, x_score)   # "A", "B", "UNDECIDED"
if result == "A": 
    result = "Cross"
elif result == "B":
    result = "X"


# ⑥ 정답 정규화
expected = normalize_label(pattern["expected"])

if result == expected:
    Pass_Fail = "PASS"
else:
    Pass_Fail = "FAIL"

print(f"--- {key} ---") 
print(f"Cross 점수: {cross_score}")
print(f"X 점수: {x_score}")

print(f"판정: {result} | expected: {expected} | {Pass_Fail}")

--- size_5_1 ---
Cross 점수: 0.9
X 점수: 0.8999999999999999
판정: UNDECIDED | expected: X | FAIL


In [186]:
t_cnt = len(data["patterns"].keys())   # test count
p_cnt = 0   # pass count
f_cnt = 0   # fail count
case_dict = {}
for key in data["patterns"].keys():
    pattern = data["patterns"][key]
    size = get_size(key)
    cross_filt = filters[size]["Cross"]
    x_filt = filters[size]["X"]
    cross_score = mac(pattern["input"], cross_filt)
    x_score = mac(pattern["input"], x_filt)
    result = judge(cross_score, x_score)   
    if result == "A": 
        result = "Cross"
    elif result == "B":
        result = "X"
    expected = normalize_label(pattern["expected"])
    if result == expected:
        Pass_Fail = "PASS"
        p_cnt += 1
    else:
        Pass_Fail = "FAIL"
        f_cnt += 1
        if result == "UNDECIDED":
            case_dict[key] = "동점(UNDECIDED)"
        else:
            case_dict[key] = "판정 불일치"

    print(f"\n--- {key} ---") 
    print(f"Cross 점수: {cross_score}")
    print(f"X 점수: {x_score}")
    print(f"판정: {result} | expected: {expected} | {Pass_Fail}")

print(f"총 테스트: {t_cnt}개")
print(f"통과: {p_cnt}개")
print(f"실패: {f_cnt}개")
print("실패 케이스")
for key, value in case_dict.items():
    print(f"{key}: {value}")
    
  


--- size_5_1 ---
Cross 점수: 0.9
X 점수: 0.8999999999999999
판정: UNDECIDED | expected: X | FAIL

--- size_5_2 ---
Cross 점수: 8.9
X 점수: 0.1
판정: Cross | expected: Cross | PASS

--- size_13_1 ---
Cross 점수: 0.3
X 점수: 14.700000000000008
판정: X | expected: X | PASS

--- size_13_2 ---
Cross 점수: 7.499999999999997
X 점수: 7.5
판정: UNDECIDED | expected: Cross | FAIL

--- size_25_1 ---
Cross 점수: 4.9
X 점수: 4.899999999999999
판정: UNDECIDED | expected: X | FAIL

--- size_25_2 ---
Cross 점수: 52.9
X 점수: 0.1
판정: Cross | expected: Cross | PASS
총 테스트: 6개
통과: 3개
실패: 3개
실패 케이스
size_5_1: 동점(UNDECIDED)
size_13_2: 동점(UNDECIDED)
size_25_1: 동점(UNDECIDED)


In [173]:
print(t_cnt)
print(p_cnt)
print(f_cnt)
print(case_dict)

6
3
3
{'size_5_1': '동점 판정', 'size_13_2': '동점 판정', 'size_25_1': '동점 판정'}


[5, 13, 25]


In [196]:
sizes = sorted([3] + list(filters.keys()))
print(f"{'크기':<8}{'평균 시간(ms)':>14}{'연산 횟수':>10}")
print("-------------------------------------")
for n in sizes:
    nxn = [[1.0 for _ in range(n)] for _ in range(n)]
    size = f"{n}x{n}"
    cal_vol = n*n
    _, avg_t = measure_mac(nxn,nxn)
    print(f"{size:<8}{avg_t:>14.4f}{cal_vol:>10}")

크기           평균 시간(ms)     연산 횟수
-------------------------------------
3x3             0.0006         9
5x5             0.0012        25
13x13           0.0061       169
25x25           0.0216       625


In [191]:
new = [[1.0 for _ in range(3)] for _ in range(3)]

In [192]:
new

[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]